In [ ]:
import os
# Set this to "1" to disable torch.compile globally
# os.environ["TORCH_COMPILE_DISABLE"] = "1"
import torch
import torch.nn as nn

import numpy as np
from typing import NamedTuple
import matplotlib.pyplot as plt
from scipy.stats import gamma, poisson

from models.utils import build_warmup_epochs
from models.inn import RealNVP,RealNVPSummary
from models.regressionNetwork import RegressionNetwork, train_regression_network

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

from tqdm import tqdm
import torch.nn.utils as utils

In [ ]:
dataset = np.load("dataset.npz")

seed = 67
print(dataset)
foot = dataset["foot"]

com_trace = dataset["com"]
com = com_trace[:,com_trace.shape[1] // 2] # get middle position

print(f"{foot.shape=}")
print(f"{com.shape=}")
print(f"{com_trace.shape=}")
# flatten foot
foot_reshape = foot.reshape((foot.shape[0],-1)) # flatten(1)?

In [ ]:


print(f"{foot.shape}")
print(f"{com.shape}")

assert foot.shape[0] == com.shape[0]
N = foot.shape[0]
indices = np.arange(N)

# 2. Shuffle the indices in-place using a generator for better control

rng = np.random.default_rng(seed)
rng.shuffle(indices)

# 3. Calculate the split point
split_point = int(N * 0.8)

# 4. Slice the shuffled array into two vectors
train_indices = indices[:split_point]
test_indices = indices[split_point:]

# declare final arrays
train_foot = foot[train_indices]
train_com = com[train_indices]


test_foot = foot[test_indices]
test_com = com[test_indices]
print(f"{test_foot.shape=}, {test_com.shape=}")

In [ ]:
for frame in range(com_trace.shape[1]):
    plt.hist(com_trace[:,frame,0],bins= 25)
    plt.show()

In [ ]:
for frame in range(foot.shape[1]):
    plt.hist(foot[:,frame,0],bins= 25)
    plt.show()


![GIF:](https://upload.wikimedia.org/wikipedia/commons/7/71/ChessPawnSpecialMoves.gif "chess")


In [ ]:
# create dataloaders:
from torch.utils.data import DataLoader,TensorDataset
dtype = torch.float32
train_tensor_com = torch.tensor(train_com).to(dtype)
train_tensor_foot = torch.tensor(train_foot).to(dtype)
test_tensor_com = torch.tensor(test_com).to(dtype)
test_tensor_foot = torch.tensor(test_foot).to(dtype)

train_foot_mean = train_tensor_foot.mean()
train_foot_std = train_tensor_foot.std()
train_com_mean = train_tensor_com.mean(0,keepdim=True)
train_com_std = train_tensor_com.std(0,keepdim=True)

print(train_foot_mean,train_foot_std,train_com_mean,train_com_std)

train_tensor_com = (train_tensor_com - train_com_mean) / train_com_std
train_tensor_foot = (train_tensor_foot - train_foot_mean) / train_foot_std
test_tensor_com = (test_tensor_com - train_com_mean) / train_com_std
test_tensor_foot = (test_tensor_foot - train_foot_mean) / train_foot_std


train_datatensor = TensorDataset(train_tensor_com,train_tensor_foot)
test_datatensor = TensorDataset(test_tensor_com,test_tensor_foot)

def create_dataloaders(batch_size,num_workers = 11):
    train_loader = DataLoader(
                train_datatensor,
                shuffle=True,
                batch_size=batch_size,
                num_workers=num_workers,
                pin_memory=True,
                persistent_workers=True,
                drop_last=True # otherwise, some instances are weighed higher
            )

    test_loader = DataLoader(
                test_datatensor,
                shuffle=False, # unnecesarry
                batch_size=batch_size, # OPT: increase
                num_workers=max(num_workers // 2,8),
                pin_memory=True,
                persistent_workers=True,
                
            )
    return train_loader,test_loader


In [ ]:
# print input sizes for model
train_loader = create_dataloaders(32)[0]
com_test,foot_test = next(iter(train_loader))
print(foot_test.shape,com_test.shape)


In [ ]:
from torch import GradScaler
from models.inn import CouplingBlock
@torch.compile()
def calculate_loss(model: RealNVPSummary, output):
    loss = torch.zeros((), requires_grad=True, device=device)
    # for block in model.realNVP.blocks: # use for summary network
    for block in model.realNVP.blocks:
        block: CouplingBlock
        loss = loss - block.block_det
    loss += output.pow(2).sum() / 2  # sum over all axis
    return loss / output.size(0)  # mean over batch


def train_inn_cond(
    model: RealNVP,
    train_loader: torch.utils.data.DataLoader,
    test_loader: torch.utils.data.DataLoader,
    optim: torch.optim.Optimizer,
    scaler: GradScaler,
    lr_scheduler: torch.optim.lr_scheduler.LRScheduler,
    epochs: int,
    history,
    batch_size=128,
):
    """
    model: A inn
    train_set_fn: a function that returns X,Y
    with:
    X: the hidden parameter that we want a posterior for later.
    Y: the condition (like observations that will be available at inference time)

    # scaler = GradScaler("cuda", enabled=(device.type == 'cuda'))
    """
    history["train_loss"] = history.get("train_loss", [])
    history["test_loss"] = history.get("test_loss", [])
    model.train()
    pbar = tqdm(range(epochs), desc="Training",leave=True)
    for epoch in pbar:
        train_epoch_loss = torch.zeros((1,), requires_grad=False, device=device)
        test_epoch_loss = torch.zeros((1,), requires_grad=False, device=device)
        train_epoch_length = len(train_loader)
        test_epoch_length = len(test_loader)
        # for now, define a epoch as 100 iterations, because
        # we have no train set size yet.
        
        for com, foot in train_loader:
            com: torch.Tensor
            foot: torch.Tensor
            X, Y = com.to(device), foot.to(device)  # TODO: nonblocking
            optim.zero_grad()
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                #print(X.shape,Y.shape)
                output = model.forward(X, Y)
                # y is used for condition, x is input (see docstring)
                loss = calculate_loss(model, output)
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            scaler.step(optim)
            scaler.update()
            train_epoch_loss += loss.mean()

        with torch.no_grad():
            model.eval()
            for com, foot in test_loader:
                com: torch.Tensor
                foot: torch.Tensor
                X, Y = com.to(device), foot.to(device)  # TODO: nonblocking
                with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                    #print(X.shape,Y.shape)
                    output = model.forward(X, Y)
                    loss = calculate_loss(model, output)


                test_epoch_loss += loss.mean()
            # print(loss.mean().item())

        model.train()
        train_avg_epoch_loss = (train_epoch_loss / train_epoch_length).item()
        test_avg_epoch_loss = (test_epoch_loss / test_epoch_length).item()
        # print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_epoch_loss:.4f}", end="\r")
        pbar.set_postfix(train_loss=train_avg_epoch_loss,test_loss=test_avg_epoch_loss)
        history["train_loss"].append(train_avg_epoch_loss)
        history["test_loss"].append(test_avg_epoch_loss)
        lr_scheduler.step()
    model.eval()
    return


In [ ]:
# TODO: normalize inputs
input_size = 3  #
observable_size = 39 * 6  # maybe use the spacial structure of the data
reduced_obs_size = 12

# model = RealNVP(
#     input_size=input_size,
#     hidden_size=hidden_size,
#     blocks=num_coup_blocks,
#     condition_size=observable_size,
# )
model = RealNVPSummary(
    input_size=input_size,
    condition_size=observable_size,
    reduced_condition_size=reduced_obs_size,
    s_hidden=128,
    s_layers=3,
    r_hidden=32,
    r_blocks=8,
)
# model = RealNVPSummary(input_size=input_size,condition_size=observable_size,reduced_condition_size=reduced_obs_size,s_hidden=256,s_layers=3,r_hidden=32,r_blocks=6)


model.to(device)
model = torch.compile(model, mode="max-autotune", fullgraph=True)

num_epochs = 15
lr_warmup_epochs = 4
init_lr = 3e-3
weight_decay = 5e-5
optimizer = torch.optim.AdamW(model.parameters(), init_lr, weight_decay=weight_decay)
history = {}

batch_size = 64
train_loader, test_loader = create_dataloaders(batch_size)

scaler = torch.GradScaler("cuda", enabled=(device.type == "cuda"))
lr_scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, build_warmup_epochs(lr_warmup_epochs, num_epochs)
)


train_inn_cond(
    model,
    train_loader=train_loader,
    test_loader=test_loader,
    optim=optimizer,
    scaler=scaler,
    lr_scheduler=lr_scheduler,
    epochs=num_epochs,
    history=history,
    batch_size=None,
)

In [ ]:
import matplotlib.pyplot as plt
model.eval()
# 1. Set a clean style
plt.style.use('seaborn-v0_8-muted') # or 'ggplot'
plt.figure(figsize=(10, 5), dpi=100)

# 2. Plot with better aesthetics
plt.plot(np.array(history["train_loss"][:]), 
         color='#1f77b4',       # A nice professional blue
         linewidth=1,           # Slightly thicker line
         label='Training Loss')

plt.plot(np.array(history["test_loss"][:]), 
         color="#c50488",       # A nice professional blue
         linewidth=1,           # Slightly thicker line
         label='Test Loss')


# 3. Add context and labels
plt.title('Model Training Progress', fontsize=14, pad=15)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)

# 4. Clean up the "frame"
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(frameon=True)
plt.tight_layout()
#plt.yscale("log")
plt.show()

In [ ]:
# 1. Set a clean style
plt.style.use('seaborn-v0_8-muted') # or 'ggplot'
plt.figure(figsize=(10, 5), dpi=100)

# 2. Plot with better aesthetics
plt.plot(-np.array(history["test_loss"][len(history["test_loss"])//4 * 3: ]), 
         color='#1f77b4',       # A nice professional blue
         linewidth=2,           # Slightly thicker line
         label='test Loss')

# 3. Add context and labels
plt.title('Last forth of test loss', fontsize=14, pad=15)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)

# 4. Clean up the "frame"
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(frameon=True)
plt.tight_layout()
plt.yscale("log")
plt.show()

In [ ]:
# import torch

def compute_mmd(x, y, sigma=1.0):
    """
    Computes the Maximum Mean Discrepancy (MMD) between two sets of samples.
    Args:
        x: Tensor of shape [n, d] (Samples from Distribution P)
        y: Tensor of shape [m, d] (Samples from Distribution Q)
        sigma: Bandwidth of the RBF kernel
    """
    # 1. Compute distance matrix (n+m, n+m)
    z = torch.cat([x, y], dim=0)
    # Efficient pairwise distance calculation: ||a-b||^2 = ||a||^2 + ||b||^2 - 2<a,b>
    dist_matrix = torch.cdist(z, z, p=2)**2
    
    # 2. Apply Gaussian Kernel
    kernel_matrix = torch.exp(-dist_matrix / (2 * sigma**2))
    
    # 3. Extract sub-matrices
    n = x.size(0)
    m = y.size(0)
    
    k_xx = kernel_matrix[:n, :n]
    k_yy = kernel_matrix[n:, n:]
    k_xy = kernel_matrix[:n, n:]
    
    # 4. Compute MMD^2 (Unbiased estimator)
    # Subtracting the diagonal (self-distances) for k_xx and k_yy
    mmd2 = (k_xx.sum() - n) / (n * (n - 1)) + \
           (k_yy.sum() - m) / (m * (m - 1)) - \
           2 * k_xy.mean()
           
    return mmd2

# Example Usage
x = torch.randn(100, 2)  # Mean 0
y = torch.randn(100, 2) + 0.5  # Mean 0.5
loss = compute_mmd(x, y)
print(f"MMD: {loss.item():.4f}")

In [ ]:
with torch.no_grad():
    latent = model.forward(train_tensor_com.to(device),train_tensor_foot.to(device)).cpu()
    print(latent.shape)
    print(f"Train Set: {latent.std(0)=}")
    (print(f"{torch.cov(latent.T)=}"))
    plt.scatter(latent[:,0],latent[:,1])
    plt.show()
    plt.hist(latent[:,0])
    plt.show()
    plt.hist(latent[:,1])
    plt.show()

In [ ]:

with torch.no_grad():
    latent = model.forward(test_tensor_com.to(device),test_tensor_foot.to(device)).cpu()
    print(f"Test Set: {latent.std(0)=}")
    plt.scatter(latent[:,0],latent[:,1])
    plt.show()
    plt.hist(latent[:,0])
    plt.show()
    plt.hist(latent[:,1])
    plt.show()

In [ ]:
# visualize the dataset com
_ = plt.hist(com[:,0],bins=25)

In [ ]:
res  = model.reverse(torch.normal(0,1,(1000,3)).to(device) ,train_tensor_foot[:1000].to(device) )
print(res.shape)

In [ ]:
corrected = res.cpu() * train_com_std + train_com_mean
corrected = corrected.detach()
print(corrected.shape)

In [ ]:
_ = plt.hist(corrected.T,bins=25)
plt.legend(["x","y","z"])


In [ ]:
print(com[:1000].shape)
print(corrected.numpy().shape)
plt.hist((com[:1000] - corrected.numpy()).flatten() )

In [ ]:
plt.scatter(np.arange(20),corrected[:20,0])
plt.scatter(np.arange(20),com[:20,0])
#TODO: copy code from model_realnvp_single.ipynb to visualize confidence

In [ ]:
filename_model_params = "saved_model_test.model"
filename_model_hyperparams = "saved_model_test.def"
torch.save(model.state_dict(), filename_model_params)
model_hyperparams_dict = {
    "input_size": model.realNVP.input_size,
    "condition_size": model.summary.input_size,
    "reduced_condition_size": model.realNVP.condition_size,
    "s_hidden": model.summary.encoder_layers[0].weight.shape[0],
    "s_layers": model.summary.layers,
    "r_hidden": model.realNVP.blocks[0].scale_net[0].weight.shape[0],
    "r_blocks": len(model.realNVP.blocks),
}
torch.save(model_hyperparams_dict,filename_model_hyperparams)


In [ ]:
init_dict = torch.load(filename_model_hyperparams)
print(init_dict)
loaded_dict = torch.load(filename_model_params)
model = RealNVPSummary(**init_dict)
model = torch.compile(model)
model: RealNVPSummary
model.load_state_dict(loaded_dict)